# City LTS: fresh build 2017–2022 (bike LTS 1–4) + 2026 (bike LTS 1–4 + car LTS 5)

**Works for any French commune in your official GeoJSON.** Set `CITY` in section 1 and run all cells.
For another city, change `CITY` and run all again: each city gets its own output folder `lts_output/<city>/<RUN_NAME>/`.
Run one city at a time (the Overpass servers refuse parallel downloads).

This notebook ignores every earlier cache and pickle. Everything is downloaded again into a **new run folder**.

* **2017–2022:** historical OSM snapshot (31 Dec of each year), bicycle network, LTS 1–4.
* **2026:** current OSM. Bicycle network LTS 1–4, **plus** car roads that are not in the bicycle network as **LTS 5**.
  An LTS 5 road is flagged `mandatory_upgrade_to_lts1=True` when it is dangerous (speed ≥ 50 km/h, ≥ 4 lanes, or primary/secondary class)
  and cycling is not prohibited on it (motorway/trunk/`bicycle=no` get `bicycle_prohibited=True` instead).
* **Simplification (every year):** single undirected `nx.Graph` (worst LTS kept on parallel edges), largest connected component.
  A degree-2 junction is removed when both edges have the same LTS; the new edge length is the **sum** of the two.
  For LTS 5, the merged edge is flagged for an upgrade to LTS 1 if **at least one** of the two edges was flagged.
* **Metrics** are computed on the simplified graph: cycling length and largest-connected-component length at LTS ≤ 1, 2, 3, 4.
  **LTS 5 is never counted** (it is only a candidate for future upgrades).
* The simplified 2026 graph (bikes + cars) used for the metrics is saved as a pickle.

Files needed next to this notebook: `overpass_robust.py`, `grenoble_lts_graph.py`, `grenoble_workflow_fixed.py`,
`grenoble_2026_extension.py`, `grenoble_lts_historical.py`, `grenoble_simplification_exact.py`, `grenoble_metrics_exponential.py`, `plot_2026_lts.py`
(the module names say "grenoble" but the code works for any polygon).

## 1. Settings

In [3]:
from pathlib import Path
import pickle
import geojson
from shapely.geometry import shape
import networkx as nx
import pandas as pd
from IPython.display import display

from overpass_robust import download_graph, OVERPASS_ENDPOINTS
from grenoble_lts_graph import configure_osmnx, annotate_graph, largest_weak_component, normalize_polygon, _values
from grenoble_lts_historical import normalize_snapshot_date
from grenoble_workflow_fixed import prepare_bike
from grenoble_2026_extension import _missing_car_network
from grenoble_simplification_exact import simplify_lts_graph, check_graph, plot_topology
from grenoble_metrics_exponential import annual_metrics, exponential_weights

# ---- CHANGE THESE ---------------------------------------------------------
CITY = "Lyon"
CITY_INSEE = None   # only needed if several communes share the name, e.g. "35238" for Rennes
result_file = Path(r"C:\Users\P323041\OneDrive - University of Groningen\Documents\Important\Poyosky\21-10-2026\Bebote_cloud\communes-20220101.json")

# A NEW run name = a completely fresh build (new download cache, new outputs).
# Keep the same name to resume a run that stopped halfway (finished years are not downloaded again).
RUN_NAME = "fresh_2026-10-21_Lyon"
REBUILD = False        # True = also ignore pickles inside THIS run folder and redo every year
# ---------------------------------------------------------------------------

HISTORICAL_YEARS = list(range(2017, 2023))
CURRENT_YEAR = 2026
TIMEOUT = 600          # seconds the Overpass server may work on one query
MATCH = "way"          # "way": car road is LTS 5 only if its OSM way is not in the bike graph
                       # "endpoints": original rule (can duplicate streets the bike graph split differently)
ENDPOINTS = OVERPASS_ENDPOINTS   # tried in order if one times out

with result_file.open("r", encoding="utf-8") as f:
    gj = geojson.load(f)
matches = [f for f in gj["features"] if f["properties"].get("nom") == CITY
           and (CITY_INSEE is None or str(f["properties"].get("insee", f["properties"].get("code"))) == str(CITY_INSEE))]
if not matches:
    raise ValueError(f"No commune named {CITY!r} in {result_file.name} (check spelling/accents)")
if len(matches) > 1:
    codes = [m["properties"].get("insee", m["properties"].get("code")) for m in matches]
    raise ValueError(f"{len(matches)} communes are named {CITY!r}; set CITY_INSEE to one of {codes}")
poly = shape(matches[0]["geometry"])
if not poly.is_valid:
    poly = poly.buffer(0)
if poly.is_empty:
    raise ValueError("The selected city polygon is empty")

SLUG = CITY.lower().replace(" ", "_").replace("'", "")
RUN = Path("lts_output") / SLUG / RUN_NAME
CACHE = RUN / "osm_http_cache"
RUN.mkdir(parents=True, exist_ok=True)
polygon = normalize_polygon(poly)
configure_osmnx()   # keeps cycleway/maxspeed/lanes/bicycle tags needed by the classifier
print("City:", CITY, "| Polygon:", poly.geom_type, "| Valid:", poly.is_valid, "| Bounds:", poly.bounds)
print("Output:", RUN.resolve())

City: Lyon | Polygon: Polygon | Valid: True | Bounds: (4.771813, 45.707367, 4.898377, 45.808263)
Output: C:\Users\P323041\OneDrive - University of Groningen\Documents\Important\Poyosky\21-10-2026\Bebote_cloud\lts_output\lyon\fresh_2026-10-21_Lyon


## 2. Build functions

In [4]:
def save(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def load(path):
    with path.open("rb") as f:   # only open your own pickle files
        return pickle.load(f)

def snapshot(year):
    """Historical years -> 31 Dec timestamp; 2026 -> None (current OSM)."""
    return None if year == CURRENT_YEAR else normalize_snapshot_date(str(year))

def get_bike(year):
    """Download the bicycle network, classify raw LTS, map to LTS 1-4."""
    path = RUN / str(year) / f"{SLUG}_bike_{year}_prepared.pkl"
    if path.exists() and not REBUILD:
        return load(path)
    date = snapshot(year)
    G = download_graph(polygon, "bike", date=date, server_timeout=TIMEOUT, endpoints=ENDPOINTS,
                       cache_folder=CACHE / (date[:10] if date else "current"))
    steps = [(u, v, k) for u, v, k, d in G.edges(keys=True, data=True) if "steps" in _values(d, "highway")]
    G.remove_edges_from(steps)
    G.remove_nodes_from(list(nx.isolates(G)))
    G = annotate_graph(G)                       # raw lts_score 0,1,2,2.5,3,4,5
    bike = prepare_bike(largest_weak_component(G))   # -> key/lts 1..4
    bike.graph["year"] = year
    save(bike, path)
    return bike

def get_cars(year):
    path = RUN / str(year) / f"{SLUG}_car_{year}_raw.pkl"
    if path.exists() and not REBUILD:
        return load(path)
    date = snapshot(year)
    cars = download_graph(polygon, "drive", date=date, server_timeout=TIMEOUT, endpoints=ENDPOINTS,
                          cache_folder=CACHE / (date[:10] if date else "current"))
    save(cars, path)
    return cars

def build_year(year):
    out = RUN / str(year) / f"{SLUG}_lts_{year}_SIMPLE.pkl"
    if out.exists() and not REBUILD:
        print(f"[{year}] loaded {out}")
        return load(out)
    bike = get_bike(year)
    if year == CURRENT_YEAR:
        source = _missing_car_network(bike, get_cars(year), match=MATCH)   # LTS 1-4 + LTS 5
        save(source, RUN / str(year) / f"{SLUG}_lts5_{year}_pre_simplification.pkl")
    else:
        source = bike
    H = simplify_lts_graph(source)   # same-LTS degree-2 removal, summed length, OR of upgrade flag
    H.graph.update(year=year, city=CITY, snapshot=snapshot(year) or "current OSM")
    check = check_graph(H)
    assert type(H) is nx.Graph and check["connected"]
    assert check["remaining_contractible_degree2_nodes"] == 0
    levels = {d["key"] for _, _, d in H.edges(data=True)}
    if year == CURRENT_YEAR:
        assert levels <= {1, 2, 3, 4, 5} and 5 in levels, "No LTS 5 roads in 2026"
    else:
        assert levels <= {1, 2, 3, 4}
    save(H, out)
    print(f"[{year}] saved {out}")
    return H

## 3. Build all seven networks
Each year is saved as soon as it finishes. If a year fails, rerun this cell: finished years load from disk.

In [ ]:
metric_graphs, failed = {}, {}
for year in HISTORICAL_YEARS + [CURRENT_YEAR]:
    try:
        metric_graphs[year] = build_year(year)
    except Exception as exc:
        failed[year] = repr(exc)
        print(f"[{year}] FAILED: {exc}")
print("Done:", sorted(metric_graphs), "| Failed:", failed)

[overpass] bike | https://overpass-api.de/api | attempt 1 | server timeout 600s | date=2017-12-31T23:59:59Z
[overpass] failed after 1s -> ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry y
[overpass] bike | https://overpass-api.de/api | attempt 2 | server timeout 600s | date=2017-12-31T23:59:59Z
[overpass] failed after 0s -> ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry y
[overpass] bike | https://overpass.private.coffee/api | attempt 1 | server timeout 600s | date=2017-12-31T23:59:59Z


## 4. Metrics on the simplified graphs (LTS 5 excluded)
Length = sum of LTS 1–4 edge lengths. LCC_t = length of the largest connected component (by nodes) using only edges with LTS ≤ t. The normalized values divide by a fixed reference (default: the largest cycling length of the seven years).

In [ ]:
if failed:
    raise RuntimeError(f"Rerun section 3 first; missing years: {sorted(failed)}")

KAPPA = 1.0
metrics = annual_metrics(metric_graphs, kappa=KAPPA, reference_length_km=None)
cols = ["year", "cycling_length_km", "normalized_network_length",
        "lcc_lts1_km", "lcc_lts2_km", "lcc_lts3_km", "lcc_lts4_km",
        "weighted_lcc_normalized", "lts5_edges", "lts5_mandatory_edges"]
display(metrics[cols].round(4))
print("Weights:", exponential_weights(KAPPA))

csv_path = RUN / f"{SLUG}_2017_2022_2026_SIMPLIFIED_metrics.csv"
txt_path = RUN / f"{SLUG}_2017_2022_2026_SIMPLIFIED_metrics.txt"
metrics.to_csv(csv_path, index=False)
with txt_path.open("w", encoding="utf-8") as f:
    f.write(f"{CITY} simplified cycling metrics, fresh build {RUN_NAME}\n")
    f.write("2017-2022: bicycle LTS1-4 simplified graphs (historical OSM, 31 Dec)\n")
    f.write("2026: bicycle LTS1-4 + missing car roads LTS5 simplified together; LTS5 excluded from metrics\n")
    f.write(f"Car matching rule: {MATCH}; kappa={KAPPA}\n\n")
    f.write(metrics.to_string(index=False, float_format=lambda x: f"{x:.6f}") + "\n")
print("Saved:", csv_path, "and", txt_path)

## 5. The saved 2026 graph (bikes + cars) used for the metrics

In [ ]:
path_2026 = RUN / str(CURRENT_YEAR) / f"{SLUG}_lts_{CURRENT_YEAR}_SIMPLE.pkl"
G_2026 = load(path_2026)
assert nx.utils.graphs_equal(G_2026, metric_graphs[CURRENT_YEAR])   # same graph the metrics used

lts5 = [d for _, _, d in G_2026.edges(data=True) if d["key"] == 5]
print("File:", path_2026.resolve())
print(f"{G_2026.number_of_nodes():,} nodes, {G_2026.number_of_edges():,} edges")
print("Edges per LTS:", dict(sorted(check_graph(G_2026)["lts_counts"].items())))
print(f"LTS 5 edges: {len(lts5):,} | must upgrade to LTS 1: {sum(d['mandatory_upgrade_to_lts1'] for d in lts5):,}"
      f" | cycling prohibited: {sum(bool(d.get('bicycle_prohibited')) for d in lts5):,}")
print(f"LTS 5 length: {sum(d['distance'] for d in lts5)/1000:.2f} km, of which mandatory "
      f"{sum(d['distance'] for d in lts5 if d['mandatory_upgrade_to_lts1'])/1000:.2f} km")

## 6. Plots of the simplified networks
Straight lines between the remaining nodes (topology view). Grey = LTS 5, purple = LTS 5 that must be upgraded to LTS 1.

In [ ]:
for year, G in sorted(metric_graphs.items()):
    print(f"{CITY} {year}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    fig = plot_topology(G, show_nodes=False)

## 7. 2026 before vs after simplification
Left: merged bike + car graph before simplification. Right: the saved simplified graph. Blue dots = degree-2 nodes; after simplification they should only remain where the LTS changes. Zoom in one panel and the other follows; hover an edge midpoint for its LTS and length.

In [ ]:
from plot_2026_lts import plot_before_after, leftover_degree2

G_before = load(RUN / str(CURRENT_YEAR) / f"{SLUG}_lts5_{CURRENT_YEAR}_pre_simplification.pkl")
fig = plot_before_after(G_before, G_2026, title=f"{CITY} {CURRENT_YEAR}")
leftover_degree2(G_2026)

## 8. Where everything is saved

In [ ]:
for p in sorted(RUN.rglob("*")):
    if p.is_file() and "osm_http_cache" not in p.parts:
        print(f"{p.stat().st_size/1e6:8.2f} MB  {p.resolve()}")